# Benchmark the three ETW algorithms

This notebook compares:

1. **Direct cubic:** explicitly scans every possible final-block start.
2. **Exact quadratic:** evaluates the same recurrence using upper envelopes.
3. **Adaptive multiscale:** solves a small coarse problem, lifts the path, and refines exactly inside sparse corridors.

Two regimes are included:

- **Smooth, distinctive signal:** moderate warp and low noise; multiscale refinement should remain narrow.
- **Noisy signal with stronger warp:** stronger warp and more noise; this is realistic rather than adversarial, but the coarse path is less precise and adaptive refinement should do more work.

Change **`N_MAX`** below to any power of two. The notebook benchmarks every power of two from `N_MIN` through `N_MAX`, produces slide-ready figures, and saves a shareable `*_RETURN_THIS.zip` bundle.

The cubic and exact quadratic methods have separate caps because they become expensive before the multiscale method. They still appear together over their common feasible range.

In [ ]:
from pathlib import Path
import sys
import time

from IPython.display import HTML, Image, display

# Main input: any power of two, e.g. 512, 1024, 2048, or 4096.
N_MAX = 1024
N_MIN = 64

# Timing repetitions. Three is a good default for a presentation benchmark.
REPEATS = 3
SEED = 356

# Safety caps for the dense methods. The multiscale method runs through N_MAX.
CUBIC_MAX_N = min(N_MAX, 512)
QUADRATIC_MAX_N = min(N_MAX, 2048)

# Multiscale settings.
COARSEST_SIZE = 32
INITIAL_RADIUS = 4
STABILITY_TOLERANCE = 1.0e-4
MAX_ATTEMPTS = 8

# Also compute a fixed-radius score as an accuracy diagnostic. It is not shown
# as a fourth headline timing method.
FIXED_RADIUS_DIAGNOSTIC = True
WARM_UP_NUMBA = True

# Find the repository root whether Jupyter started in the root or examples/.
ROOT = Path.cwd().resolve()
if not (ROOT / "hellinger_etw_multiscale.py").exists():
    ROOT = ROOT.parent
if not (ROOT / "hellinger_etw_multiscale.py").exists():
    raise RuntimeError("Start Jupyter from the repository root, then reopen this notebook.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.benchmark_three_methods import (
    BenchmarkConfig,
    DEFAULT_REGIMES,
    estimate_dense_peak_gb,
    run_benchmark,
    summary_html,
    warm_numba,
    write_benchmark_bundle,
)

config = BenchmarkConfig(
    n_max=N_MAX,
    n_min=N_MIN,
    seed=SEED,
    repeats=REPEATS,
    cubic_max_n=CUBIC_MAX_N,
    quadratic_max_n=QUADRATIC_MAX_N,
    coarsest_size=COARSEST_SIZE,
    initial_radius=INITIAL_RADIUS,
    adaptive=True,
    stability_tolerance=STABILITY_TOLERANCE,
    max_attempts=MAX_ATTEMPTS,
    fixed_radius_diagnostic=FIXED_RADIUS_DIAGNOSTIC,
    use_numba=True,
)

print(f"Repository root: {ROOT}")
print(f"Benchmark sizes: powers of two from {N_MIN} through {N_MAX}")
print(f"Direct cubic through N={CUBIC_MAX_N}")
print(f"Exact quadratic through N={QUADRATIC_MAX_N}")
print(
    "Approximate dense peak at the quadratic cap: "
    f"{estimate_dense_peak_gb(QUADRATIC_MAX_N):.2f} GB"
)

## Benchmark conventions

For each regime and size, the **same generated instance** is used by all methods.

The wall-clock figure is end-to-end:

- the dense methods pay the measured vectorized dense-similarity construction time plus their DP time;
- the multiscale method is timed around the complete hierarchy, including sparse similarity evaluation.

Numba compilation is warmed on a tiny problem before measurement. The shaded bands in the timing plot are the interquartile range over repeated runs.

The adaptive method is the headline multiscale method. A fixed-radius run is also recorded as a diagnostic so that the accuracy/work figure can show what adaptive widening is buying.

In [ ]:
if WARM_UP_NUMBA:
    warm_numba(
        DEFAULT_REGIMES,
        seed=SEED,
        coarsest_size=min(16, COARSEST_SIZE),
        initial_radius=max(1, INITIAL_RADIUS // 2),
    )
    print("Numba warm-up complete.")

In [ ]:
start = time.perf_counter()
run = run_benchmark(config, DEFAULT_REGIMES, progress=True)
elapsed = time.perf_counter() - start
print(f"\nComplete benchmark time: {elapsed:.2f} seconds")

In [ ]:
display(HTML(summary_html(run)))

print("\nObserved log-log timing slopes (fit to the last up to four points):")
for fit in run.scaling_fits:
    print(
        f"  {fit['regime_label']:<34s}  {fit['method_label']:<22s} "
        f"slope={fit['slope']:.2f}  "
        f"(N={fit['n_min']}..{fit['n_max']})"
    )

In [ ]:
output_dir = ROOT / "user_runs" / f"three_method_benchmark_N{N_MAX}_seed{SEED}"
notebook_path = ROOT / "examples" / "run_three_method_benchmark.ipynb"
files = write_benchmark_bundle(
    run,
    output_dir=output_dir,
    prefix=f"etw_three_method_benchmark_N{N_MAX}_seed{SEED}",
    notebook_path=notebook_path,
)

print("Benchmark regimes:")
display(Image(filename=str(files["regime_figure"])))
print("Wall-clock scaling:")
display(Image(filename=str(files["timing_figure"])))
print("Multiscale accuracy and work:")
display(Image(filename=str(files["quality_figure"])))

print("\nFiles written:")
for label, file_path in files.items():
    print(f"  {label:24s} {file_path.relative_to(ROOT)}")
print("\nPlease send back this file:")
print(files["return_zip"])

## Reading the plots

- On the **wall-clock** figure, the cubic line ends at `CUBIC_MAX_N` and the exact quadratic line ends at `QUADRATIC_MAX_N`. The multiscale line continues through `N_MAX`.
- The upper row of the **accuracy/work** figure compares score gaps with the exact quadratic solution where that solution was run. Values at roughly `1e-16` are floating-point zero.
- The lower row shows the total number of pair cells evaluated across all multiscale levels and adaptive attempts, as a percentage of the finest full `N²` grid. The dotted curve is the accepted radius at the finest level.

Adaptive score stability and boundary freedom are diagnostics, not a proof of unrestricted exactness. The raw CSV and JSON files retain the full stopping diagnostics.